In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install wandb evaluate


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import wandb
import evaluate


In [ ]:
path_dataset = '/content/drive/MyDrive/TFG/titles_data.jsonl'

with open(path_dataset, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]
df = pd.DataFrame(data)

print(f"Total dataset shape: {df.shape}")


In [ ]:
# 1. Group Shuffle Split strategy based on 'id' -> Train, Eval, Test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42) # 70% train, 30% temp
train_idx, temp_idx = next(gss1.split(df, groups=df['group_id']))

df_train = df.iloc[train_idx].copy()
df_temp = df.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42) # 15% eval, 15% test
eval_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['group_id']))

df_eval = df_temp.iloc[eval_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

print(f"Train shape: {df_train.shape}, Evalu shape: {df_eval.shape}, Test shape: {df_test.shape}")

# Convert to HF Datasets
dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'eval': Dataset.from_pandas(df_eval),
    'test': Dataset.from_pandas(df_test)
})


In [ ]:
model_path = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: "IA", 1: "Human"}
label2id = {"IA": 0, "Human": 1}

def preprocess_function(examples):
    tokenized_inputs = tokenizer(examples["title"], truncation=True)
    tokenized_inputs["labels"] = examples["is_real"]
    return tokenized_inputs

tokenized_data = dataset_dict.map(preprocess_function, batched=True)


In [ ]:
# 3. Robust metric computation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if np.isnan(predictions).any():
        print("NaN found in predictions before probability conversion! Replacing with 0.")
        predictions = np.nan_to_num(predictions)

    predicted_classes = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_classes, average='macro', zero_division=0)
    acc = accuracy_score(labels, predicted_classes)

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall
    }


In [ ]:
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'eval/f1_macro',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-4
        },
        'per_device_train_batch_size': {
            'values': [8, 16]
        },
        'num_train_epochs': {
            'values': [2, 3, 4, 5]
        },
        'weight_decay': {
            'values': [0.0, 0.01, 0.1]
        }
    }
}

sweep_id = wandb.sweep(sweep_config, project="TFG-RoBERTa-Detector")


In [ ]:
def train_sweep():
    with wandb.init() as run:
        config = wandb.config

        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=2, id2label=id2label, label2id=label2id
        )

        training_args = TrainingArguments(
            output_dir="./temp_checkpoints",
            learning_rate=config.learning_rate,
            per_device_train_batch_size=config.per_device_train_batch_size,
            per_device_eval_batch_size=8,
            num_train_epochs=config.num_train_epochs,
            weight_decay=config.weight_decay,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1_macro",
            greater_is_better=True,
            report_to="wandb"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_data["train"],
            eval_dataset=tokenized_data["eval"],
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics
        )

        trainer.train()


In [ ]:
print("Starting Hyperparameter Search (Sweep)...")
wandb.agent(sweep_id, function=train_sweep, count=5)


In [ ]:
# 5. Final Training with Best Hyperparameters
# Sustituir con los mejores valores del Sweep tras ejecutarlo
best_config = {
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "num_train_epochs": 4,
    "weight_decay": 0.01
}

print("Starting final training con la mejor configuracion...")
final_model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2, id2label=id2label, label2id=label2id
)

final_args = TrainingArguments(
    output_dir="./final_roberta_model",
    learning_rate=best_config["learning_rate"],
    per_device_train_batch_size=best_config["per_device_train_batch_size"],
    per_device_eval_batch_size=8,
    num_train_epochs=best_config["num_train_epochs"],
    weight_decay=best_config["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    report_to="none" # Ya no reportamos el final al sweep agent
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["eval"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

final_trainer.train()


In [ ]:
# 6. Evaluation on Test Set
test_results = final_trainer.evaluate(tokenized_data["test"])
print("Test Set Results:", test_results)


In [ ]:
# 7. Predictions, Visualizations, and Stats
predictions, labels, metrics = final_trainer.predict(tokenized_data["test"])
predicted_classes = np.argmax(predictions, axis=1)

# Classification Report
print("Classification Report:")
print(classification_report(labels, predicted_classes, target_names=["IA", "Human"], zero_division=0))

# Confusion Matrix
cm = confusion_matrix(labels, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["IA", "Human"], yticklabels=["IA", "Human"])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - RoBERTa Test Set')
plt.show()


In [ ]:
# Training history visualization
log_history = final_trainer.state.log_history

train_loss = [log['loss'] for log in log_history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in log_history if 'eval_loss' in log]
eval_f1 = [log['eval_f1_macro'] for log in log_history if 'eval_f1_macro' in log]

epochs = range(1, len(train_loss) + 1)

plt.figure(figsize=(12, 5))

# Plot Losses
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, label='Train Loss', marker='o')
plt.plot(range(1, len(eval_loss) + 1), eval_loss, label='Eval Loss', marker='o')
plt.title('Training and Evaluation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot Eval F1
plt.subplot(1, 2, 2)
if eval_f1:
    plt.plot(range(1, len(eval_f1) + 1), eval_f1, label='Eval F1 Macro', marker='o', color='green')
    plt.title('Evaluation F1 Score')
    plt.xlabel('Epochs')
    plt.ylabel('F1 Score')
    plt.legend()

plt.tight_layout()
plt.show()
